In [35]:
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.messages import HumanMessage
import requests

In [36]:

@tool
def get_conversion_factor(base_currency : str, target_currency : str) -> float:
    
    """
    This function fetches the currency conversion factor between a given base currency and target currency
    """
    url = f'https://v6.exchangerate-api.com/v6/99e78127e062abaaf97eb3f8/pair/{base_currency}/{target_currency}'
    response = requests.get(url)

    return response.json()


In [37]:
@tool
def convert(base_currency_value: int, conversion_cost: float):
    """
    given a currency conversion rate this function calculate the target currency value from a given base currency value
    """

    return base_currency_value* conversion_cost

In [38]:
result = get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})
result['conversion_rate']

90.9715

In [39]:
response = convert.invoke({'base_currency_value':12,'conversion_cost':90.9715})
response

1091.6580000000001

# tools binding

In [40]:
llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")
llm_with_tools = llm.bind_tools([get_conversion_factor,convert])

In [41]:
llm_with_tools

RunnableBinding(bound=ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x0000024246023280>, default_metadata=(), model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_conversion_factor', 'description': 'This function fetches the currency conversion factor between a given base currency and target currency', 'parameters': {'properties': {'base_currency': {'type': 'string'}, 'target_currency': {'type': 'string'}}, 'required': ['base_currency', 'target_currency'], 'type': 'object'}}}, {'type': 'function', 'fun

# tool calling

In [44]:
messages = [HumanMessage('What is conversion factor between USD and INR, and can you convert the 10 USD into INR')]
messages

[HumanMessage(content='What is conversion factor between USD and INR, and can you convert the 10 USD into INR', additional_kwargs={}, response_metadata={})]

In [46]:
llm_with_tools.invoke(messages).tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'b0edff27-111f-4f46-befd-87148bba907c',
  'type': 'tool_call'}]